# The ensemble explained

This notebook explains the approach of `02_final_ensembles.ipynb`, so that you can walk Daniel through it and defend it. It trains nothing that matters: toy data shows the concepts, and saved predictions show the real results.

**How to use it:** press Run All, then read from top to bottom. It takes about a minute. **The code cells only draw pictures**, so you do not need to explain them.

| Section | Question it answers |
|---|---|
| 1 | What is the approach in one picture? |
| 2 | What goes in, and how are we scored? |
| 3 | How is the data prepared? |
| 4 | How do logistic regression, kNN and the MLP work? |
| 5 | How are they combined, and why does that help? |
| 6 | Can the confidence be trusted? |
| 7 | How do we measure the score honestly? |
| 8 | What are the results? |
| 9 | What was tried outside `02`, and what did it decide? |
| 10 | How does this differ from Daniel's stack? |
| 11 | What questions should you be ready for? |

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import ListedColormap
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch, Rectangle
from sklearn.datasets import make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, LIGHT = "#0b0b0b", "#52514e", "#e4e3df"
COLORS = [BLUE, ORANGE, AQUA]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})


def softmax(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def box(ax, x, y, w, h, text, fill="white", edge=MUTED, color=INK, size=10):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.01,rounding_size=0.08",
                                facecolor=fill, edgecolor=edge, linewidth=1.2))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", color=color, fontsize=size)


def arrow(ax, start, end, color=MUTED):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=12, color=color,
                                 linewidth=1.2, shrinkA=0, shrinkB=0))


# Toy data used throughout: two curved classes and one round cluster
moons, moon_labels = make_moons(n_samples=300, noise=0.17, random_state=0)
blob, _ = make_blobs(n_samples=150, centers=[[2.9, 1.0]], cluster_std=0.3, random_state=0)
toy_X = np.vstack([moons, blob])
toy_y = np.concatenate([moon_labels, np.full(150, 2)])


def plot_regions(ax, proba, title, X=toy_X, y=toy_y, legend=False):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                         np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))
    classes = proba(np.c_[xx.ravel(), yy.ravel()]).argmax(axis=1).reshape(xx.shape)
    ax.contourf(xx, yy, classes, levels=np.arange(-0.5, 3), cmap=ListedColormap(COLORS), alpha=0.16)
    for k in range(3):
        ax.scatter(*X[y == k].T, s=11, color=COLORS[k], edgecolor="white", linewidth=0.3, label=f"class {k + 1}")
    ax.set(title=title, xticks=[], yticks=[])
    if legend:
        ax.legend(frameon=False, loc="lower right", fontsize=9)

## 1. The approach in one picture

**The idea:** three models that think differently each give a probability for all 25 industries. We average those probabilities. The highest average is the prediction, and its value is the confidence.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.3))
box(ax, 0.0, 2.0, 1.7, 1.0, "Company\ndescription\n(text)")
box(ax, 2.2, 2.0, 1.7, 1.0, "Embedding\n768 numbers\n(given to us)", fill="#cde2fb", edge=BLUE)
box(ax, 2.2, 0.2, 1.7, 0.9, "Country and\nfinancials\n(optional)", fill="#f4f3ef")
box(ax, 4.9, 3.4, 2.2, 0.9, "Logistic regression", fill="white", edge=BLUE)
box(ax, 4.9, 2.05, 2.2, 0.9, "kNN\n(embedding only)", fill="white", edge=BLUE)
box(ax, 4.9, 0.7, 2.2, 0.9, "MLP", fill="white", edge=BLUE)
box(ax, 8.1, 2.05, 1.5, 0.9, "Average the\nprobabilities", fill="#cde2fb", edge=BLUE)
box(ax, 10.2, 2.05, 1.8, 0.9, "Prediction\nand confidence", fill=BLUE, edge=BLUE, color="white")
arrow(ax, (1.72, 2.5), (2.18, 2.5))
for y in (3.85, 2.5, 1.15):
    arrow(ax, (3.92, 2.5), (4.87, y))
    arrow(ax, (7.12, y), (8.07, 2.5))
arrow(ax, (3.92, 0.65), (4.87, 1.0))
arrow(ax, (3.92, 0.85), (4.87, 3.5))
arrow(ax, (9.62, 2.5), (10.18, 2.5))
ax.text(7.75, 0.55, "each model gives\n25 probabilities", ha="left", color=MUTED, fontsize=9)
ax.set(xlim=(-0.1, 12.1), ylim=(0, 4.6))
ax.axis("off")
plt.show()

## 2. What goes in, and how are we scored?

### Embeddings

We do not get the company text, only its **embedding**: 768 numbers produced by a language model. Descriptions with similar meaning get vectors that point in a similar direction.

We compare two embeddings with **cosine similarity**, the cosine of the angle between them. It ignores length and looks only at direction:

$$\cos\theta = \frac{a \cdot b}{\|a\|\,\|b\|}, \qquad \text{cosine distance} = 1 - \cos\theta$$

The picture uses 2 dimensions. The real space has 768, but the idea is the same.

In [ ]:
words = {"gold miner": (0.92, 0.38), "copper miner": (0.80, 0.60), "software firm": (-0.45, 0.89)}
fig, ax = plt.subplots(figsize=(6.5, 4))
for (name, (x, y)), color in zip(words.items(), [BLUE, BLUE, ORANGE]):
    ax.annotate("", xy=(x, y), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=color, lw=2))
    ax.text(x * 1.05, y * 1.07, name, color=INK, ha="left" if x > 0 else "right")
a, b, c = [np.array(v) for v in words.values()]
ax.set(xlim=(-1.05, 1.35), ylim=(-0.05, 1.08), xticks=[], yticks=[], title="Similar meaning = small angle",
       xlabel=f"cosine distance: gold vs copper {1 - a @ b:.2f}, gold vs software {1 - a @ c:.2f}")
ax.spines[["left", "bottom"]].set_visible(False)
plt.show()

### The score

$$\text{score} = \text{weighted F1} \times (1 - \text{Brier})$$

**Weighted F1** measures whether predictions are right. F1 balances precision and recall for each industry. "Weighted" means each industry counts by its number of companies, so big industries matter more.

**Brier** measures whether the confidence is honest:

$$\text{Brier} = \frac{1}{n} \sum_i (\text{confidence}_i - \text{correct}_i)^2$$

Here `correct` is 1 for a right prediction and 0 for a wrong one.

The picture shows why honesty wins. Suppose that among predictions like this one, 75% are right. The expected penalty (dark line) is lowest at a confidence of exactly 0.75. Claiming more or claiming less both cost points.

**Two consequences:**
- The score can never exceed F1, because `1 - Brier` is at most 1.
- With honest confidence, Brier is about `c(1 - c)` averaged over predictions, so it stays above zero unless the model is certain. A model with 76% accuracy cannot reach a score of 0.85.

In [ ]:
c = np.linspace(0, 1, 201)
accuracy = 0.75
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(c, (1 - c) ** 2, color=BLUE, linewidth=2, label="penalty if the prediction is right")
ax.plot(c, c ** 2, color=ORANGE, linewidth=2, label="penalty if the prediction is wrong")
ax.plot(c, accuracy * (1 - c) ** 2 + (1 - accuracy) * c ** 2, color=INK, linewidth=2.5,
        label="expected penalty when 75% are right")
ax.plot(accuracy, accuracy * 0.25 ** 2 + 0.25 * 0.75 ** 2, "o", color=INK, markersize=8)
ax.annotate("lowest at confidence = 0.75", xy=(0.75, 0.1875), xytext=(0.36, 0.52), color=INK,
            arrowprops=dict(arrowstyle="-", color=MUTED))
ax.set(xlabel="confidence we report", ylabel="squared penalty", xlim=(0, 1), ylim=(0, 1))
ax.legend(frameon=False, loc="upper center", ncol=1, bbox_to_anchor=(0.5, 1.02))
plt.show()

## 3. How is the data prepared?

### Deduplication

The raw file has 31,706 rows, and 4,578 of them repeat an earlier embedding. One single embedding appears 4,359 times, and 99.7% of those rows are Materials, so it is probably an empty or boilerplate description.

| Step | Rows |
|---|---|
| Raw data | 31,706 |
| After removing duplicate embeddings | 27,128 |

**Why this matters:**
- The private test set is cleaned, so we must evaluate on cleaned data. Keeping the duplicates inflated every score by about 0.06, because those rows are trivially easy.
- Duplicates must go **once, before any split**. Otherwise the same company could sit in both train and test, and the model would be tested on data it has already seen. This is called **leakage**.

### Fit on training rows only

Anything learned from data must be learned from the training rows alone: the mean of the embeddings, scaling factors, percentiles and medians. If test rows helped compute them, information about the test set would leak into training. In `02`, `ExtraFeatures.fit` and `center` follow this rule.

### Country and financial features

These are optional extras for LR and the MLP. The cleaning is explained in `03_financial_analysis.ipynb`. In short: zero revenue gets a flag, impossible values become missing, gaps get a median plus a missing flag, and outliers are shrunk with a signed log and clipping.

**kNN never gets them**, because adding 0/1 columns to the embedding would distort the cosine distance between descriptions.

## 4. The three models

### 4.1 Logistic regression (LR)

**Idea:** each industry gets its own weight vector. A company's score for an industry is a weighted sum of its features. **Softmax** then turns the 25 scores into probabilities:

$$\text{score}_k = w_k \cdot x + b_k \qquad\qquad P(k \mid x) = \frac{e^{\text{score}_k}}{\sum_j e^{\text{score}_j}}$$

Training adjusts the weights so that the true industry gets a high probability (it minimizes cross-entropy).

Because the scores are linear, **the boundaries between classes are straight lines**. In the picture, LR handles the round cluster on the right easily, but it cuts straight through the two curved classes.

In [ ]:
lr_toy = LogisticRegression(C=1).fit(toy_X, toy_y)
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.4, 1]})
plot_regions(left, lr_toy.predict_proba, "Logistic regression: straight boundaries", legend=True)

scores = np.array([2.0, 1.0, -1.0])
probabilities = softmax(scores)
names = ["Materials", "Capital Goods", "Banks"]
position = np.arange(3)
right.barh(position + 0.19, scores, height=0.34, color=LIGHT, label="raw score")
right.barh(position - 0.19, probabilities, height=0.34, color=BLUE, label="after softmax")
for i in range(3):
    right.text(max(scores[i], 0) + 0.05, i + 0.19, f"{scores[i]:.1f}", va="center", color=MUTED)
    right.text(probabilities[i] + 0.05, i - 0.19, f"{probabilities[i]:.2f}", va="center", color=INK)
right.set_yticks(position, names)
right.invert_yaxis()
right.axvline(0, color=MUTED, linewidth=1)
right.set_title("Softmax turns scores into probabilities that sum to 1")
right.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

**What `C` does.** Large weights make probabilities extreme, so LR adds a penalty on them. `C` sets how weak that penalty is:

- **Small `C`:** strong penalty, cautious and blurry probabilities.
- **Large `C`:** weak penalty, sharp probabilities that follow the training data closely.

The shading shows confidence, so darker means more confident. The grid search found `C=3` (score 0.622), better than the default `C=1` (0.613). Going higher gave no real gain.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
xx, yy = np.meshgrid(np.linspace(-2, 4, 250), np.linspace(-1.2, 2.2, 250))
for ax, C, label in zip(axes, [0.01, 100], ["strong penalty", "weak penalty"]):
    model = LogisticRegression(C=C, max_iter=3000).fit(toy_X, toy_y)
    confidence = model.predict_proba(np.c_[xx.ravel(), yy.ravel()]).max(axis=1).reshape(xx.shape)
    image = ax.imshow(confidence, origin="lower", extent=(-2, 4, -1.2, 2.2), cmap="Blues", vmin=0.33, vmax=1, aspect="auto")
    ax.set(title=f"C = {C}: {label}", xticks=[], yticks=[])
fig.colorbar(image, ax=axes, label="confidence", shrink=0.85)
plt.show()

### 4.2 k-nearest neighbours (kNN)

**Idea:** there is no real training. kNN stores all companies. For a new company it finds the `k` most similar ones and lets them vote. A share of the votes is its probability.

Here, `k=15`: if 9 of the 15 neighbours are Materials, then P(Materials) is about 0.6.

**Strength:** it finds small local groups that a straight boundary misses, for example a niche of very similar companies.

**Weakness:** it copies its neighbours, so it inherits their label mistakes, and it can give an industry a probability of exactly 0.

In [ ]:
query = np.array([0.5, 0.25])
distance = np.linalg.norm(toy_X - query, axis=1)
nearest = np.argsort(distance)[:15]
radius = distance[nearest[-1]]
votes = np.bincount(toy_y[nearest], minlength=3)

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.5, 1]})
for k in range(3):
    left.scatter(*toy_X[toy_y == k].T, s=12, color=COLORS[k], alpha=0.35, linewidth=0, label=f"class {k + 1}")
left.scatter(*toy_X[nearest].T, s=42, c=[COLORS[k] for k in toy_y[nearest]], edgecolor=INK, linewidth=0.8, zorder=3)
for point in toy_X[nearest]:
    left.plot(*zip(query, point), color=MUTED, linewidth=0.6, zorder=2)
left.add_patch(Circle(query, radius, fill=False, linestyle="--", edgecolor=MUTED))
left.scatter(*query, marker="*", s=300, color=INK, edgecolor="white", zorder=4, label="new company")
left.set(title="The 15 nearest companies vote", xticks=[], yticks=[], aspect="equal")
left.legend(frameon=False, loc="lower right", fontsize=9)

right.bar(range(3), votes / 15, color=COLORS)
for i, value in enumerate(votes):
    right.text(i, value / 15 + 0.02, f"{value}/15 = {value / 15:.2f}", ha="center", color=INK)
right.set_xticks(range(3), ["class 1", "class 2", "class 3"])
right.set(ylim=(0, 1), ylabel="probability", title="Votes become probabilities")
plt.tight_layout()
plt.show()

**Choosing `k`.** A small `k` trusts a few neighbours, so noise creates jagged little islands. A large `k` averages too widely and blurs real structure. On the real data, `k` between 10 and 30 worked best.

In [ ]:
hard_X, hard_y = make_moons(n_samples=300, noise=0.32, random_state=3)   # harder, overlapping classes
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, k, text in zip(axes, [1, 15, 150], ["too jumpy", "a good balance", "too smooth"]):
    model = KNeighborsClassifier(n_neighbors=k).fit(hard_X, hard_y)
    plot_regions(ax, model.predict_proba, f"k = {k}: {text}", X=hard_X, y=hard_y)
plt.tight_layout()
plt.show()

**Distance weights.** A plain vote treats the 15th neighbour like the 1st. We use `weights = exp(-10 * d)`, so a near neighbour counts far more than a distant one, as the picture shows. This also makes the result less sensitive to the exact `k`.

**Centering.** All embeddings tend to point in a broadly similar direction. `center` subtracts the average embedding and rescales to length 1, so the cosine distance focuses on what differs between companies. To be honest about its effect: on the real data, centering alone added only about 0.001.

In [ ]:
d = np.linspace(0, 0.6, 200)
rng = np.random.default_rng(1)
cloud = rng.normal([3, 3], [0.7, 0.25], size=(14, 2))
cloud = cloud @ np.array([[0.8, -0.6], [0.6, 0.8]]) @ np.array([[0.71, 0.71], [-0.71, 0.71]]).T

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].plot(d, np.exp(-10 * d), color=BLUE, linewidth=2, label="exp(-10 d), used here")
axes[0].plot(d, np.ones_like(d), color=MUTED, linewidth=2, linestyle="--", label="plain vote")
axes[0].set(xlabel="cosine distance to neighbour", ylabel="weight of its vote", ylim=(0, 1.1), title="Closer neighbours count more")
axes[0].legend(frameon=False, loc="center right")

for ax, points, title in [(axes[1], cloud, "Before: angles are all tiny"),
                          (axes[2], cloud - cloud.mean(axis=0), "After centering: angles spread out")]:
    unit = points / np.linalg.norm(points, axis=1, keepdims=True)
    for u in unit:
        ax.plot([0, u[0]], [0, u[1]], color=BLUE, linewidth=1.3)
    ax.add_patch(Circle((0, 0), 1, fill=False, edgecolor=LIGHT))
    ax.scatter(*unit.T, s=16, color=BLUE, zorder=3)
    ax.set(xlim=(-1.2, 1.2), ylim=(-1.2, 1.2), aspect="equal", xticks=[], yticks=[], title=title)
    ax.spines[["left", "bottom"]].set_visible(False)
plt.tight_layout()
plt.show()

### 4.3 Multi-layer perceptron (MLP)

**Idea:** an MLP is logistic regression on features it learns itself. A hidden layer sits between the input and the output.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.6))
layers = [(1.0, 6, "768 inputs\n(the embedding)"), (4.2, 7, "512 hidden neurons\n(ReLU)"), (7.4, 5, "25 outputs\n(softmax)")]
centers = []
for x, n, label in layers:
    ys = np.linspace(0.6, 3.8, n)
    ys = np.delete(ys, n // 2)   # leave a gap for the dots
    centers.append([(x, y) for y in ys])
    ax.text(x, 2.2, "...", ha="center", va="center", fontsize=15, color=MUTED, rotation=90)
    ax.text(x, -0.25, label, ha="center", va="center", color=INK)
for layer_a, layer_b in zip(centers, centers[1:]):
    for a in layer_a:
        for b in layer_b:
            ax.plot(*zip(a, b), color=LIGHT, linewidth=0.7, zorder=1)
for layer, color in zip(centers, [BLUE, ORANGE, AQUA]):
    for point in layer:
        ax.add_patch(Circle(point, 0.17, facecolor=color, edgecolor="white", zorder=2))
ax.text(9.0, 2.2, "highest probability\n= prediction\n\nits value\n= confidence", va="center", color=INK)
ax.set(xlim=(0, 10.8), ylim=(-0.7, 4.2), aspect="equal", title="Every neuron connects to the whole next layer: about 406,000 weights")
ax.axis("off")
plt.show()

**What one neuron does:** it computes a weighted sum, as in LR, and passes it through **ReLU**, `max(0, value)`.

**Why ReLU matters:** without it, stacking layers of weighted sums gives just another weighted sum, so the network would still be linear. ReLU adds a bend. Adding many bent lines can approximate curves, as the right picture shows.

**How it learns:**
1. **Forward pass:** a batch of companies goes through the network.
2. **Loss:** cross-entropy measures how wrong the probabilities are.
3. **Backpropagation:** the chain rule computes how much each weight contributed to the error.
4. **Update:** the Adam optimizer moves each weight slightly to reduce the loss.
5. **Repeat** for many passes over the data, called epochs.

In [ ]:
x = np.linspace(-3, 3, 400)
relu = lambda v: np.maximum(0, v)
bent = [relu(x + 1.5), -2 * relu(x), 2 * relu(x - 1.5)]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].plot(x, relu(x), color=BLUE, linewidth=2.5)
axes[0].set(title="ReLU: max(0, value)", xlabel="input", ylabel="output")
for curve, color, label in zip(bent, COLORS, ["neuron 1", "neuron 2", "neuron 3"]):
    axes[1].plot(x, curve, color=color, linewidth=2, label=label)
axes[1].set(title="Three neurons, each with one bend", xlabel="input")
axes[1].legend(frameon=False, loc="lower left")
axes[2].plot(x, sum(bent), color=INK, linewidth=2.5)
axes[2].set(title="Their sum: a shape no straight line can make", xlabel="input")
plt.tight_layout()
plt.show()

**The MLP can follow curves.** On the same toy data, it wraps around the two curved classes where LR had to cut straight through.

In [ ]:
mlp_toy = MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-2, max_iter=3000, random_state=0).fit(toy_X, toy_y)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
plot_regions(axes[0], lr_toy.predict_proba, "Logistic regression: straight")
plot_regions(axes[1], mlp_toy.predict_proba, "MLP: curved", legend=True)
plt.tight_layout()
plt.show()

**Stopping it from memorizing.** Your MLP has about 406,000 weights but only about 24,000 training companies, so it could memorize the training data. If that happens, the training score keeps rising while the score on unseen data falls. The picture shows this on noisy toy data.

| Setting in `02` | What it does |
|---|---|
| `StandardScaler` | Puts all inputs on a similar scale, so training runs smoothly |
| `alpha=1e-2` | Penalizes large weights. Larger `alpha` means stronger, the opposite direction of `C` |
| `early_stopping=True` | Holds back 10% of the training rows and stops when their score stops improving |
| 3 seeds averaged | Training starts from random weights, so averaging 3 networks removes much of that luck |

In [ ]:
X_noisy, y_noisy = make_moons(n_samples=500, noise=0.3, random_state=1)
flip = np.random.default_rng(1).random(len(y_noisy)) < 0.2     # 20% wrong labels
y_noisy = np.where(flip, 1 - y_noisy, y_noisy)
train, held_out = np.arange(350), np.arange(350, 500)

net = MLPClassifier(hidden_layer_sizes=(300, 300), alpha=0, learning_rate_init=0.003, random_state=0)
train_curve, held_out_curve = [], []
for epoch in range(400):
    net.partial_fit(X_noisy[train], y_noisy[train], classes=[0, 1])
    train_curve.append(net.score(X_noisy[train], y_noisy[train]))
    held_out_curve.append(net.score(X_noisy[held_out], y_noisy[held_out]))
smooth = pd.Series(held_out_curve).rolling(15, min_periods=1).mean()
best = int(smooth.idxmax())

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(train_curve, color=BLUE, linewidth=2, label="training rows")
ax.plot(smooth, color=ORANGE, linewidth=2, label="held-out rows")
ax.axvline(best, color=MUTED, linestyle="--")
ax.text(best + 6, 0.59, "early stopping\nwould stop near here", color=INK)
ax.set(xlabel="epochs", ylabel="accuracy", ylim=(0.55, 1.0), title="More training is not always better")
ax.legend(frameon=False, loc="upper left", ncol=2)
plt.show()

### Comparing the three models

| | Logistic regression | kNN | MLP |
|---|---|---|---|
| **Decides by** | Weighted sum per industry | Vote of similar companies | Learned features, then weighted sums |
| **Boundary** | Straight | Local and irregular | Curved |
| **Training** | Fast | None, it stores the data | Slowest |
| **Prediction** | Fast | Slowest, it searches all rows | Fast |
| **Main setting** | `C` | `k`, distance weights | Layer size, `alpha`, early stopping |
| **Real score alone** | 0.622 | 0.620 | 0.626 |

## 5. Combining the models

```python
P = (P_lr + P_knn + P_mlp) / 3     # soft vote
prediction = P.argmax(axis=1)      # industry with the highest average
confidence = P.max(axis=1)         # that average
```

**Soft vote** averages probabilities. A **hard vote** would only count each model's top choice, and would throw away how sure each model is.

### Why does averaging help?

Each model's errors have two parts: a shared part, such as wrong labels, and a personal part that comes from how that model works. Averaging keeps what they agree on and shrinks personal errors. **This only works if the models make different mistakes**, so three different model types help more than three similar ones.

In the picture, the soft vote follows the curve found by kNN and the MLP, while LR pulls it toward a simpler boundary.

In [ ]:
knn_toy = KNeighborsClassifier(n_neighbors=15).fit(toy_X, toy_y)
models = {"Logistic regression": lr_toy.predict_proba, "kNN": knn_toy.predict_proba, "MLP": mlp_toy.predict_proba}
models["Soft vote of all three"] = lambda Z: sum(f(Z) for f in list(models.values())[:3]) / 3

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, (name, proba) in zip(axes, models.items()):
    plot_regions(ax, proba, name)
plt.tight_layout()
plt.show()

### On the real data

The next cells use the out-of-fold predictions from the earlier 5-fold run: every company was predicted by models that never saw it. They are saved in `claude_experiments/cache/dedup`, outside the repository. If that folder is missing, these cells print a message and skip.

In [ ]:
CACHE = "../../claude_experiments/cache/dedup/"
REAL_DATA = os.path.exists(CACHE + "oof_lr.npy")

if REAL_DATA:
    industries = (pd.read_pickle("../../data/df_train.pkl")
                  .drop_duplicates(subset="business_description_embedding")["industry"].to_numpy())
    class_names = np.unique(industries)
    y = np.searchsorted(class_names, industries)
    P_lr1, P_lr, P_knn, P_mlp = [np.load(CACHE + f"oof_{n}.npy") for n in ("lr1", "lr", "knn", "skmlp")]
    P_ensemble = (P_lr + P_knn + P_mlp) / 3


    def score_of(P, confidence=None):
        pred = P.argmax(axis=1)
        confidence = P.max(axis=1) if confidence is None else confidence
        return f1_score(y, pred, average="weighted") * (1 - np.mean((confidence - (pred == y)) ** 2))


    print(f"loaded {len(y)} companies, ensemble score {score_of(P_ensemble):.4f}")
else:
    print("Saved predictions not found: the real-data pictures will be skipped.")

**How often do the models disagree?** They give the same answer for about 4 of 5 companies. Disagreements mark the hard cases, so they also carry useful information about uncertainty.

In [ ]:
if REAL_DATA:
    lr_pred, knn_pred, mlp_pred, ensemble_pred = [P.argmax(axis=1) for P in (P_lr, P_knn, P_mlp, P_ensemble)]
    agree = (lr_pred == knn_pred) & (knn_pred == mlp_pred)
    print(f"all three agree:     {agree.mean():.1%} of companies, accuracy {(ensemble_pred == y)[agree].mean():.1%}")
    print(f"at least one differs: {(~agree).mean():.1%} of companies, ensemble accuracy {(ensemble_pred == y)[~agree].mean():.1%}")

    only_one_right = ((lr_pred == y).astype(int) + (knn_pred == y) + (mlp_pred == y)) == 1
    print(f"exactly one model right: {only_one_right.mean():.1%} of companies, "
          f"the ensemble is right on {(ensemble_pred == y)[only_one_right].mean():.1%} of those")

    # A real company where one model is right, two are wrong, and the average gets it right
    candidates = np.where(only_one_right & (ensemble_pred == y) & (P_ensemble.max(axis=1) > 0.3) & (P_ensemble.max(axis=1) < 0.5))[0]
    row = candidates[0]
    top = np.argsort(P_ensemble[row])[::-1][:4]
    short = [class_names[k][:36] for k in top]

    fig, axes = plt.subplots(1, 4, figsize=(15, 3), sharex=True, sharey=True)
    for ax, P, name in zip(axes, [P_lr, P_knn, P_mlp, P_ensemble], ["Logistic regression", "kNN", "MLP", "Average"]):
        values = P[row, top]
        ax.barh(range(4), values, color=[BLUE if k == y[row] else LIGHT for k in top])
        for i, value in enumerate(values):
            ax.text(value + 0.02, i, f"{value:.2f}", va="center", color=INK)
        ax.set_title(name + (" (right)" if P[row].argmax() == y[row] else " (wrong)"))
        ax.set_yticks(range(4), short)
        ax.set_xlim(0, 1)
    axes[0].invert_yaxis()   # the axes are shared, so invert only once
    fig.suptitle("One real company: blue is the true industry", x=0.01, ha="left")
    plt.tight_layout()
    plt.show()

In this example only the MLP is right, and the average gets there narrowly: 0.35 against 0.34. Averaging does not always rescue a prediction like this, which is why the ensemble is right on only about a quarter of the cases where exactly one model is right.

**Why equal weights?** The experiments found no reliable gain from fitted weights. Fitting weights by log-loss under-weighted kNN, because its exact-zero probabilities are heavily punished by that loss. Equal weights have nothing to overfit, and they are easy to explain.

## 6. Can the confidence be trusted?

The confidence is the average probability, unchanged. The score rewards honest confidence, so we check it.

**Reliability diagram (left):** group predictions by confidence, then compare with how often they were right. A perfect model lies on the dashed diagonal. **Right:** how many predictions fall into each confidence level.

**How to read your result:** the line runs close to the diagonal and slightly above it. So the ensemble is well calibrated, but a little **underconfident**: it says 0.73 on average and is right 0.76 of the time. Averaging contributes to this, because disagreeing models pull the highest probability down.

In [ ]:
if REAL_DATA:
    confidence = P_ensemble.max(axis=1)
    correct = P_ensemble.argmax(axis=1) == y
    bins = pd.cut(confidence, np.linspace(0, 1, 11), include_lowest=True)
    table = pd.DataFrame({"bin": bins, "confidence": confidence, "correct": correct}).groupby("bin", observed=True).agg(
        companies=("correct", "size"), mean_confidence=("confidence", "mean"), accuracy=("correct", "mean"))

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
    axes[0].plot([0, 1], [0, 1], "--", color=MUTED, label="perfectly honest")
    axes[0].plot(table["mean_confidence"], table["accuracy"], "o-", color=BLUE, linewidth=2, markersize=7, label="ensemble")
    axes[0].set(xlabel="mean confidence", ylabel="share that was right", xlim=(0, 1), ylim=(0, 1), title="Does confidence match accuracy?", aspect="equal")
    axes[0].legend(frameon=False, loc="upper left")
    axes[1].hist(confidence, bins=30, color=BLUE, edgecolor="white", linewidth=0.5)
    axes[1].set(xlabel="confidence", ylabel="companies", title="How sure is the ensemble?")
    plt.tight_layout()
    plt.show()
    print(f"mean confidence {confidence.mean():.3f} vs accuracy {correct.mean():.3f}")

### Calibration and sharpening

**Temperature scaling** rescales probabilities with one number `T`: `softmax(log(P) / T)`. `T < 1` makes them sharper and `T > 1` makes them softer. The prediction stays the same, so only Brier changes.

**Sharpening** means transforming the confidence, such as `c ** k`, to look better on a metric. Under the course's first metric, confidence-weighted F1, `c ** 2` raised the score without any real improvement. Under `F1 × (1 - Brier)` it clearly hurts, as the right picture shows at `k = 2`.

**Reading the curves:** both peak just left of the dashed line, because the ensemble is slightly underconfident. The peak is only a tiny step above the dashed line, while moving further costs much more.

**Decision:** `02` uses the plain average. Calibration added only about 0.0005, well inside the noise, so it is not worth an extra step.

In [ ]:
if REAL_DATA:
    log_P = np.log(np.clip(P_ensemble, 1e-12, 1))
    temperatures = np.linspace(0.5, 2, 31)
    powers = np.linspace(0.5, 2.5, 41)
    by_temperature = [score_of(P_ensemble, softmax(log_P / t).max(axis=1)) for t in temperatures]
    by_power = [score_of(P_ensemble, confidence ** k) for k in powers]

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8), sharey=True)
    for ax, x, curve, label, title in [(axes[0], temperatures, by_temperature, "temperature T", "Temperature scaling"),
                                       (axes[1], powers, by_power, "power k in confidence ** k", "Sharpening the confidence")]:
        ax.plot(x, curve, color=BLUE, linewidth=2)
        ax.axvline(1, color=MUTED, linestyle="--")
        ax.text(1.02, min(by_power) + 0.002, "unchanged\nconfidence", color=MUTED)
        ax.set(xlabel=label, title=title)
    axes[0].set_ylabel("score")
    plt.tight_layout()
    plt.show()

## 7. Measuring the score honestly

### 5-fold cross-validation

One split can be lucky. In 5-fold cross-validation, every company is tested exactly once, by models trained on the other 80%. The picture shows the five rounds.

| | Quick test | 5-fold cross-validation |
|---|---|---|
| Models trained | Once per model | Five times per model |
| Test rows | 10%, about 2,700 | 100%, about 27,000 |
| Noise | About ±0.01 | Much smaller on the average |
| Use it for | Trying things out | Final numbers |

**This is why your quick-test gain from country (+0.0098) must be read carefully.** The earlier 5-fold run found +0.0030 ± 0.0015.

### Paired comparison

When two models use the same folds, we compare them fold by fold. If a difference appears in all 5 folds, it is real even when it is smaller than the fold-to-fold spread, because a hard fold is hard for both models.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
for fold in range(5):
    for part in range(5):
        test = part == fold
        ax.add_patch(Rectangle((part, 4 - fold), 0.96, 0.82, facecolor=BLUE if test else LIGHT, edgecolor="white"))
        ax.text(part + 0.48, 4 - fold + 0.41, "test" if test else "train", ha="center", va="center",
                color="white" if test else MUTED, fontsize=9)
    ax.text(-0.1, 4 - fold + 0.41, f"Round {fold + 1}", ha="right", va="center", color=INK)
ax.set(xlim=(-0.9, 5), ylim=(0, 5), title="Every company is tested exactly once")
ax.axis("off")
plt.show()

## 8. Results

All scores use deduplicated data. **Keep the two groups apart**, because scores from different test rows cannot be compared directly.

**5-fold run (the reliable numbers), from the earlier run:**

In [ ]:
if REAL_DATA:
    P_lr_country, P_mlp_country = np.load(CACHE + "oof_lr_country.npy"), np.load(CACHE + "oof_skmlp_country.npy")
    P_lr_all, P_mlp_all = np.load(CACHE + "oof_lr_all.npy"), np.load(CACHE + "oof_skmlp_all.npy")
    rows = {
        "LR C=1 (baseline)": P_lr1,
        "kNN": P_knn,
        "LR C=3": P_lr,
        "MLP": P_mlp,
        "Ensemble: embedding only": P_ensemble,
        "Ensemble: + country": (P_lr_country + P_knn + P_mlp_country) / 3,
        "Ensemble: + country + financials": (P_lr_all + P_knn + P_mlp_all) / 3,
    }
    names = list(rows)[::-1]
    values = [score_of(rows[n]) for n in names]

    fig, ax = plt.subplots(figsize=(9.5, 3.8))
    ax.scatter(values, names, s=70, color=[BLUE if n.startswith("Ensemble") else MUTED for n in names], zorder=3)
    for i, value in enumerate(values):
        ax.text(value + 0.0012, i, f"{value:.3f}", va="center", color=INK)
    ax.grid(axis="x", alpha=0.25)
    ax.set(xlim=(0.605, 0.655), xlabel="score", title="Each step adds a little")
    plt.tight_layout()
    plt.show()

**Single runs (noisier, on different test rows):**

| Model | Score | Tested on |
|---|---|---|
| Your ensemble, embedding + country | 0.6521 | Your quick test, about 2,700 rows |
| Your ensemble, + country + financials | 0.6516 | Your quick test, about 2,700 rows |
| Daniel's stack | 0.6425 | About 1,350 rows of his split |
| Your ensemble, embedding only | 0.6423 | Your quick test, about 2,700 rows |
| LR C=1 (baseline) | 0.6152 | Your quick test, about 2,700 rows |

### Why not higher?

Companies with nearly identical descriptions share an industry only 79–89% of the time, depending on how close they are. The text simply does not always decide the label, so this caps any embedding-based model. Every model family ended at about 74–76% accuracy on this data.

## 9. What was tried outside `02`

These experiments ran in `claude_experiments/` and are not in the repository. The numbers are on deduplicated data unless the table says otherwise. Each row led to a decision in `02`.

### Models

| Tried | Score | Decision |
|---|---|---|
| LR, C=1 | 0.613 | Kept as the baseline |
| LR, C=3 | 0.622 | **Used** |
| kNN, k=20, 1/d weights | 0.616 | Improved by tuning |
| kNN, k=15, `exp(-10d)`, centered | 0.620 | **Used** |
| sklearn MLP, 512 units, 3 seeds | 0.626 | **Used** |
| PyTorch MLP, 1024 units, 5 seeds | 0.632 | Better, but larger and harder to explain; left out |
| RBF SVM | 0.593 | Best F1 (0.751), but poor confidence and slow; rejected |
| XGBoost on PCA-128 | 0.619 | No better than the simpler models; rejected |
| LDA with shrinkage | 0.561, or 0.608 calibrated | Poor confidence; rejected |
| Financial data only | 0.175–0.203 | Far too weak alone |

### Embedding preprocessing

| Variant | LR | kNN | Decision |
|---|---|---|---|
| Raw | 0.622 | 0.616 | **Used for LR** |
| Center + re-normalize | 0.623 | 0.617 | **Used for kNN** |
| Standardize | 0.613 | not tried | Rejected for LR |
| PCA-256 whitened | 0.611 | 0.593 | Rejected: whitening hurts |

Your MLP does use `StandardScaler`, because neural networks train better with it.

### Combining models

| Tried | Result | Decision |
|---|---|---|
| Plain soft vote, LR + kNN + MLP | 0.640 | **Used** |
| LR + kNN only | 0.637 | The MLP adds 0.003 |
| 7 LRs with different financial inputs | 0.625 | They disagree on only 2–7% of rows, so there is no diversity |
| Log-linear blends (171 combinations) | 0.645 at best | Too complex for the small gain |
| Stacking with an LR meta-model | 0.616 | Overfit in that setup |
| Greedy model selection | 0.636–0.652 | Unstable between data halves; rejected |

### Fixes for specific errors

| Tried | Gain | Decision |
|---|---|---|
| Specialist models for the most confused pairs | 0.000 | Rejected |
| Per-class bias tuning | +0.001 | Rejected |
| Isotonic or temperature calibration | +0.0003 to +0.0005 | Skipped: inside the noise |

### What the data analysis found

- **Financial information is already partly in the embedding.** From the embedding alone, you can predict the zero-revenue flag with AUC 0.94. This explains why adding financials gives so little.
- **The errors are spread out.** The 15 most confused pairs cause only 28% of the errors, so no single fix helps much.
- **The financial file only contains training ids**, so test financials may not exist. This is why `FINAL_FEATURES = None` is the default.

### Decisions that were reversed

These are worth knowing, because the old summary file still recommends the first two.

| Earlier advice | Why it was dropped |
|---|---|
| Keep duplicates in training and validation (score about 0.68) | The private test is cleaned, so those scores were inflated |
| Temperature `T=0.9` | The gain was inside the noise, so the extra step was dropped |
| `confidence ** 2` | It only helped under the old metric, and it lowers this score |
| Target of 0.85 | According to the project notes, the target changed on 2026-10-29 to above 66%, set relative to other submissions. Check the current task text |

## 10. How does this differ from Daniel's stack?

**Both combine models, but in different ways.** A soft vote uses a fixed rule: average. **Stacking** trains a **meta-model**, a small LR that takes all the base models' probabilities as input and learns how to combine them.

To train it fairly, the meta-model needs predictions on rows that each base model did not see, or it would learn to trust models that are just memorizing. This is why Daniel uses out-of-fold predictions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.7))
for ax, title, base, combiner, x in [
    (axes[0], "Yours: soft vote", ["LR (embedding)", "kNN (embedding)", "MLP (embedding)"], "Average\n(fixed rule)", False),
    (axes[1], "Daniel's: stacking", ["kNN (embedding)", "LR (embedding)", "XGBoost (financial)", "Random forest (financial)"], "Meta-model LR\n(learned rule)", True),
]:
    positions = np.linspace(3.4, 0.2, len(base))
    for name, y in zip(base, positions):
        box(ax, 0, y, 3.0, 0.62, name, edge=BLUE)
        arrow(ax, (3.03, y + 0.31), (4.47, 2.1))
    box(ax, 4.5, 1.65, 2.0, 0.9, combiner, fill="#cde2fb", edge=BLUE)
    arrow(ax, (6.53, 2.1), (7.17, 2.1))
    box(ax, 7.2, 1.7, 1.6, 0.8, "Prediction", fill=BLUE, edge=BLUE, color="white")
    ax.set(xlim=(-0.1, 8.9), ylim=(0, 4.2), title=title)
    ax.axis("off")
plt.tight_layout()
plt.show()

| | Your ensemble | Daniel's stack |
|---|---|---|
| **Base models** | LR, kNN, MLP | kNN, LR, XGBoost, random forest |
| **Financial data** | Inside LR and MLP, optional | Separate branches, required |
| **Combination** | Average, nothing to fit | Trained meta-model |
| **Without test financials** | Works, embedding-only version | Loses two of four branches |
| **Best score** | 0.645 in 5-fold, 0.652 in quick test | 0.6425 on one split |

**Current status:** the scores are too close to name a winner. They used different test rows, so a fair comparison must score both models on the same companies.

**Why choose yours for now:** it is at least as good, simpler to explain, and it still works if the private test has no financial data.

## 11. Questions to be ready for

**Why these three models?**
Because they make different kinds of mistakes: linear, local and non-linear. The ensemble gains from that difference.

**Why not weight the models?**
Fitted weights gave no reliable gain, and equal weights cannot overfit.

**Why is your confidence just the highest probability?**
The score rewards honest confidence. The reliability diagram shows it is already close, slightly underconfident, and calibration added only about 0.0005.

**Why did you not sharpen the confidence?**
A temperature near 0.9 gains only about 0.0005, which is inside the noise, and `confidence ** 2` clearly lowers the score.

**How did you pick the hyperparameters?**
LR and kNN came from grid searches with 5-fold cross-validation on deduplicated data, using the competition score. **The MLP settings were hand-picked and then checked, not searched.** Say that openly.

**Are the scores slightly optimistic?**
Yes, a little. The grids used the same folds that produced the reported scores. With small grids the effect should be small, but a truly clean estimate would need nested cross-validation.

**Why is kNN embedding-only?**
Adding 0/1 columns would distort the cosine distance between descriptions.

**Why deduplicate?**
The private test is cleaned, and leaving duplicates would inflate the score and let the same company appear in train and test.

**Do financials help?**
Only a little, about +0.003 to +0.005 in 5-fold, because the embedding already carries much of that information. Their statistical differences between industries are still real, as `03_financial_analysis.ipynb` shows.

**Does training meet the 45-minute limit?**
`02` prints the final training time, and the notebook expects a few minutes. Check that number on your run.